# Lab 2: Setup AgentCore Gateway

이 노트북에서는 Cognito + AgentCore Gateway를 설정하고, Lab 1에서 배포한 Lambda를 MCP 도구로 연결합니다.

## Architecture
```
MCP Client → [Cognito OAuth] → AgentCore Gateway → [IAM Role] → Lambda → EMR Serverless → S3 Tables
```

## Step 1: Configuration

In [ ]:
import boto3
import json
import time
import random
import string
import os

REGION = (
    boto3.session.Session().region_name
    or os.environ.get("AWS_DEFAULT_REGION")
    or os.environ.get("AWS_REGION")
)
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
FUNCTION_NAME = "fhir-mcp-server"
LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{FUNCTION_NAME}"
GATEWAY_NAME = "fhir-medical-gateway"
GATEWAY_ROLE_NAME = "fhir-agentcore-gateway-role"

iam_client = boto3.client("iam")
cognito_client = boto3.client("cognito-idp", region_name=REGION)
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)

print(f"Account: {ACCOUNT_ID}")
print(f"Lambda: {LAMBDA_ARN}")

## Step 2: Create Cognito User Pool (Inbound Auth)

AgentCore Gateway의 인바운드 인증을 위한 Cognito User Pool을 생성합니다.

In [ ]:
# Create User Pool
pool = cognito_client.create_user_pool(PoolName="fhir-mcp-gateway-pool")
USER_POOL_ID = pool["UserPool"]["Id"]
print(f"User Pool ID: {USER_POOL_ID}")

# Create domain
domain_prefix = f"fhir-mcp-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"
cognito_client.create_user_pool_domain(Domain=domain_prefix, UserPoolId=USER_POOL_ID)
TOKEN_URL = f"https://{domain_prefix}.auth.{REGION}.amazoncognito.com/oauth2/token"
print(f"Token URL: {TOKEN_URL}")

# Create resource server with scope
cognito_client.create_resource_server(
    UserPoolId=USER_POOL_ID,
    Identifier="fhir-mcp",
    Name="FHIR MCP Server",
    Scopes=[{"ScopeName": "tools", "ScopeDescription": "Access MCP tools"}]
)

# Create app client
app_client = cognito_client.create_user_pool_client(
    UserPoolId=USER_POOL_ID,
    ClientName="fhir-mcp-client",
    GenerateSecret=True,
    AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthScopes=["fhir-mcp/tools"],
    AllowedOAuthFlowsUserPoolClient=True,
)
CLIENT_ID = app_client["UserPoolClient"]["ClientId"]
CLIENT_SECRET = app_client["UserPoolClient"]["ClientSecret"]
print(f"Client ID: {CLIENT_ID}")

DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
print(f"Discovery URL: {DISCOVERY_URL}")

## Step 3: Create Gateway Service Role (Outbound Auth)

Gateway가 Lambda를 호출할 때 사용하는 IAM 역할을 생성합니다.

In [ ]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "lambda:InvokeFunction",
        "Resource": LAMBDA_ARN
    }]
}

try:
    iam_client.create_role(RoleName=GATEWAY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"Created role: {GATEWAY_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role exists: {GATEWAY_ROLE_NAME}")

iam_client.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName="InvokeLambda",
    PolicyDocument=json.dumps(inline_policy)
)

GATEWAY_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{GATEWAY_ROLE_NAME}"
print(f"Gateway Role ARN: {GATEWAY_ROLE_ARN}")
time.sleep(10)

## Step 4: Create AgentCore Gateway

In [ ]:
gateway = agentcore_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": DISCOVERY_URL,
            "allowedClients": [CLIENT_ID]
        }
    }
)

GATEWAY_ID = gateway["gatewayId"]
GATEWAY_URL = gateway["gatewayUrl"]
print(f"Gateway ID: {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")

# Wait for gateway to become active
print("Waiting for gateway to become ACTIVE...")
for _ in range(30):
    status = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
    if status == "READY":
        print(f"Gateway is READY!")
        break
    time.sleep(5)
else:
    print(f"Gateway status: {status}")

## Step 5: Add Lambda Permission

AgentCore Gateway가 Lambda를 호출할 수 있도록 리소스 기반 정책을 추가합니다.

In [ ]:
try:
    lambda_client.add_permission(
        FunctionName=FUNCTION_NAME,
        StatementId="AgentCoreGatewayInvoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
        SourceArn=f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}"
    )
    print("Lambda permission added")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists")

## Step 6: Define Tool Schemas & Add Lambda Target

13개 MCP 도구의 스키마를 정의하고 Gateway Target으로 등록합니다.

In [ ]:
TOOL_SCHEMAS = [
    {
        "name": "get_patient_summary",
        "description": "Get comprehensive patient summary including demographics, active conditions, allergies, and current medications.",
        "inputSchema": {
            "type": "object",
            "properties": {"patient_id": {"type": "string", "description": "Patient resource ID"}},
            "required": ["patient_id"]
        }
    },
    {
        "name": "search_patients",
        "description": "Search patients by name, gender, birth date range, or condition.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Patient name (partial match)"},
                "gender": {"type": "string", "description": "Gender: male or female"},
                "birth_date_from": {"type": "string", "description": "Birth date range start (YYYY-MM-DD)"},
                "birth_date_to": {"type": "string", "description": "Birth date range end (YYYY-MM-DD)"},
                "condition_code": {"type": "string", "description": "Condition keyword (e.g. diabetes, hypertension)"}
            }
        }
    },
    {
        "name": "get_encounter_history",
        "description": "Get patient encounter history with practitioner and location details.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Patient resource ID"},
                "date_from": {"type": "string", "description": "Start date (YYYY-MM-DD)"},
                "date_to": {"type": "string", "description": "End date (YYYY-MM-DD)"},
                "class_code": {"type": "string", "description": "Encounter class: AMB, IMP, EMER"}
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "get_clinical_observations",
        "description": "Get clinical observations (vitals, lab results) for a patient. Supports LOINC code filtering.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Patient resource ID"},
                "observation_code": {"type": "string", "description": "Observation keyword (e.g. glucose, blood pressure)"},
                "date_from": {"type": "string", "description": "Start date"},
                "date_to": {"type": "string", "description": "End date"}
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "get_medications",
        "description": "Get medication requests and administration records for a patient.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Patient resource ID"},
                "active_only": {"type": "boolean", "description": "Show only active medications"}
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "get_diagnosis_history",
        "description": "Get diagnosis history including conditions and related procedures.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Patient resource ID"},
                "category": {"type": "string", "description": "Condition category filter"}
            },
            "required": ["patient_id"]
        }
    },
    {
        "name": "get_claim_summary",
        "description": "Get claim and explanation of benefit summary for billing/insurance review.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "patient_id": {"type": "string", "description": "Patient resource ID"},
                "date_from": {"type": "string", "description": "Start date"},
                "date_to": {"type": "string", "description": "End date"},
                "status": {"type": "string", "description": "Claim status filter"}
            }
        }
    },
    {
        "name": "detect_care_gaps",
        "description": "Detect care gaps: missing immunizations, overdue screenings, incomplete care plans.",
        "inputSchema": {
            "type": "object",
            "properties": {"patient_id": {"type": "string", "description": "Patient resource ID"}},
            "required": ["patient_id"]
        }
    },
    {
        "name": "get_population_health_metrics",
        "description": "Get population health metrics aggregated by condition, age group, and gender.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "condition_code": {"type": "string", "description": "Condition keyword (e.g. diabetes)"},
                "age_group": {"type": "string", "description": "Age group (e.g. 60-69)"}
            }
        }
    },
    {
        "name": "list_tables",
        "description": "List available FHIR data lake tables with metadata. Filter by domain: administrative, clinical, medication, diagnostic, care, financial, document.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "domain": {"type": "string", "description": "Domain filter"}
            }
        }
    },
    {
        "name": "get_table_schema",
        "description": "Get detailed table schema with column names, types, and descriptions.",
        "inputSchema": {
            "type": "object",
            "properties": {"table_name": {"type": "string", "description": "Table name"}},
            "required": ["table_name"]
        }
    },
    {
        "name": "get_table_relationships",
        "description": "Get table relationships inferred from reference columns for JOIN queries.",
        "inputSchema": {
            "type": "object",
            "properties": {"table_name": {"type": "string", "description": "Table name (optional, all tables if omitted)"}}
        }
    },
    {
        "name": "run_custom_query",
        "description": "Execute a read-only Spark SQL query against the FHIR data lake. Only SELECT allowed. LIMIT 100 enforced. Use list_tables and get_table_schema first to understand the schema.",
        "inputSchema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Spark SQL SELECT query"}},
            "required": ["query"]
        }
    },
    {
        "name": "search_pubmed",
        "description": "Search PubMed for medical research articles. Returns title, abstract, journal, authors, publication date, and URL. Use English keywords for best results.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query (e.g. 'type 2 diabetes SGLT2 inhibitor')"},
                "max_results": {"type": "integer", "description": "Max articles to return (default: 5, max: 20)"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_pubmed_article",
        "description": "Fetch a specific PubMed article by PMID with full abstract details.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "pmid": {"type": "string", "description": "PubMed article ID (e.g. '38472622')"}
            },
            "required": ["pmid"]
        }
    }
]

print(f"Defined {len(TOOL_SCHEMAS)} tool schemas")

In [ ]:
target = agentcore_client.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="FhirMcpLambdaTarget",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": LAMBDA_ARN,
                "toolSchema": {
                    "inlinePayload": TOOL_SCHEMAS
                }
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
)

TARGET_ID = target["targetId"]
print(f"Target ID: {TARGET_ID}")
print(f"Target Status: {target['status']}")

## Step 7: Connection Details

MCP 클라이언트 (Claude Desktop, Amazon Q 등)에서 사용할 연결 정보입니다.

In [ ]:
print("=" * 60)
print("  MCP Server Connection Details")
print("=" * 60)
print(f"MCP Server URL : {GATEWAY_URL}")
print(f"Client ID      : {CLIENT_ID}")
print(f"Client Secret  : {CLIENT_SECRET}")
print(f"Token URL      : {TOKEN_URL}")
print("=" * 60)

# Save for reference
connection_info = {
    "mcp_server_url": GATEWAY_URL,
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "token_url": TOKEN_URL,
    "gateway_id": GATEWAY_ID,
    "target_id": TARGET_ID,
    "user_pool_id": USER_POOL_ID,
}
with open("connection_info.json", "w") as f:
    json.dump(connection_info, f, indent=2)
print("\nSaved to connection_info.json")

## ✅ Done!

AgentCore Gateway가 설정되었습니다. 위 연결 정보를 MCP 클라이언트에 입력하면 13개 의료 도구를 사용할 수 있습니다.

### 배포된 리소스
- **Cognito User Pool**: `fhir-mcp-gateway-pool`
- **Gateway Service Role**: `fhir-agentcore-gateway-role`
- **AgentCore Gateway**: `fhir-medical-gateway`
- **Gateway Target**: `FhirMcpLambdaTarget` (15 tools: 13 FHIR + 2 PubMed)

## Cleanup (Optional)

워크샵 종료 후 리소스를 정리합니다.

In [ ]:
# ⚠️ Uncomment to delete resources
# agentcore_client.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetIdentifier=TARGET_ID)
# agentcore_client.delete_gateway(gatewayIdentifier=GATEWAY_ID)
# iam_client.delete_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="InvokeLambda")
# iam_client.delete_role(RoleName=GATEWAY_ROLE_NAME)
# cognito_client.delete_user_pool_domain(Domain=domain_prefix, UserPoolId=USER_POOL_ID)
# cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
# lambda_client.delete_function(FunctionName=FUNCTION_NAME)
# iam_client.delete_role_policy(RoleName="fhir-mcp-lambda-role", PolicyName="EMRServerlessLivyAccess")
# iam_client.delete_role(RoleName="fhir-mcp-lambda-role")
# print("All resources deleted")